# Cambio de tamaño con Albumentations

In [ ]:
#importar bibliotecas necesarias
from pathlib import Path
import albumentations as A
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#Definir rutas
PROYECTO = Path("..")
DATA = PROYECTO / "data"
RAW = DATA / "raw"
ruta_imagen_gato = RAW / "Sphynx_176.jpg"
ruta_imagen_mascara = RAW / "Sphynx_176.png"

## Leer las imagenes

In [ ]:
imagen_bgr = cv2.imread(str(ruta_imagen_gato))
imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
mascara = cv2.imread(str(ruta_imagen_mascara), cv2.IMREAD_GRAYSCALE)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(imagen_rgb)
ax[0].set_title("Imagen Original")
ax[0].axis("off")
ax[1].imshow(mascara, cmap="gray")
ax[1].set_title("Máscara Original")
ax[1].axis("off")
plt.show()

## Transformación de imágenes

### Definición de funciones

In [ ]:
# Resize a tamaño fijo
transform_resize = A.Compose([
    A.Resize(
        height=256,
        width=256,
        interpolation=cv2.INTER_LINEAR,
        mask_interpolation=cv2.INTER_NEAREST
    )
])

# Resize proporcional + padding
transform_resize_padding = A.Compose([
    A.LongestMaxSize(max_size=256),

    A.PadIfNeeded(
        min_height=256,
        min_width=256,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        fill_mask=2,
        position="center"
    )
])

# Resize proporcional + crop
transform_crop = A.Compose([
    A.SmallestMaxSize(max_size=256),

    A.CenterCrop(
        height=256,
        width=256
    )
])



### Aplicación de las funciones

In [ ]:
# Aplicar a imagen y máscara simultáneamente
resultado_resize = transform_resize(image=imagen_rgb, mask=mascara)
resultado_padding = transform_resize_padding(image=imagen_rgb, mask=mascara)
resultado_crop = transform_crop(image=imagen_rgb, mask=mascara)


### Vizualizar

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 13))

resultados = [
    ("Resize directo", resultado_resize),
    ("Resize + Padding", resultado_padding),
    ("Resize + CenterCrop", resultado_crop)
]

for i, (nombre, resultado) in enumerate(resultados):

    # Columna izquierda: imagen RGB
    axes[i, 0].imshow(resultado["image"])
    axes[i, 0].set_title(f"{nombre} - Imagen RGB")

    # Columna derecha: máscara
    axes[i, 1].imshow(
        resultado["mask"],
        cmap="gray",
        vmin=1,
        vmax=3,
        interpolation="nearest"
    )
    axes[i, 1].set_title(f"{nombre} - Máscara")

    # Ocultar ejes
    axes[i, 0].axis("off")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()